# Modelagem supervisionada

Comparacao de modelos em ordem de complexidade (Dummy, Regressao Logistica, Random Forest, LightGBM), com o mesmo pre-processamento integrado ao Pipeline, o mesmo split e a mesma validacao cruzada.

**Criterio definido antes dos resultados:** um modelo mais complexo so substitui a Regressao Logistica se superar a ROC-AUC dela em pelo menos 2 pontos na validacao cruzada.

**Split por municipio:** treino e teste nao compartilham municipios (ver docstring de `src/evaluation/validacao.py`).

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

from src.preprocessing.montagem import carregar_base
from src.preprocessing.features import separar_xy
from src.modeling.pipeline import criar_modelos, SEMENTE
from src.evaluation.validacao import dividir_treino_teste, comparar_modelos

X, y, w, g = separar_xy(carregar_base())
(Xtr, ytr, wtr, gtr), (Xte, yte, wte, gte) = dividir_treino_teste(X, y, w, g)

# Amostra do treino para comparar modelos (o modelo escolhido sera retreinado no treino completo)
amostra = Xtr.sample(300_000, random_state=SEMENTE).index
Xa, ya, wa, ga = Xtr.loc[amostra], ytr.loc[amostra], wtr.loc[amostra], gtr.loc[amostra]
print(f"Amostra para CV: {len(Xa):,} alunos de {ga.nunique():,} municipios")

Amostra para CV: 300,000 alunos de 4,348 municipios


In [2]:
cv_municipio = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
modelos = criar_modelos()

res_municipio = comparar_modelos(modelos, Xa, ya, wa, cv_municipio, ga, "por municipio")
res_municipio.round(3)

,modelo,split,roc_auc_media,roc_auc_dp,recall_nao_alfab,segundos
0,dummy,por municipio,0.500,0.000,0.000,2.842
1,logistica,por municipio,0.664,0.014,0.380,5.601
2,random_forest,por municipio,0.664,0.014,0.344,35.079
3,lightgbm,por municipio,0.664,0.013,0.391,10.913


In [3]:
# Demonstracao do leakage: o mesmo LightGBM com split aleatorio (municipios nos dois lados)
cv_aleatorio = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
res_aleatorio = comparar_modelos({"lightgbm": modelos["lightgbm"]}, Xa, ya, wa, cv_aleatorio,
                                 rotulo_split="aleatorio (com leakage)")

resultados_cv = pd.concat([res_municipio, res_aleatorio], ignore_index=True)
resultados_cv.to_csv("../reports/cv_comparacao_modelos.csv", index=False)
resultados_cv.round(3)

,modelo,split,roc_auc_media,roc_auc_dp,recall_nao_alfab,segundos
0,dummy,por municipio,0.500,0.000,0.000,5.649
1,logistica,por municipio,0.664,0.014,0.380,5.944
2,random_forest,por municipio,0.664,0.014,0.344,34.949
3,lightgbm,por municipio,0.664,0.013,0.391,11.165
4,lightgbm,aleatorio (com leakage),0.681,0.002,0.404,22.391


In [3]:
from src.modeling.pipeline import ESPACOS_BUSCA
from src.evaluation.validacao import otimizar

busca_log = otimizar(modelos["logistica"], ESPACOS_BUSCA["logistica"],
                     Xa, ya, wa, ga, cv_municipio, n_iter=4)
busca_lgbm = otimizar(modelos["lightgbm"], ESPACOS_BUSCA["lightgbm"],
                      Xa, ya, wa, ga, cv_municipio, n_iter=12)

print("Logistica, melhor:", busca_log.loc[0, "params"], f"AUC {busca_log.loc[0, 'mean_test_score']:.4f}")
print("LightGBM,  melhor:", busca_lgbm.loc[0, "params"], f"AUC {busca_lgbm.loc[0, 'mean_test_score']:.4f}")
busca_lgbm.head()

Logistica, melhor: {'modelo__C': 0.01} AUC 0.6638
LightGBM,  melhor: {'modelo__reg_lambda': 0.0, 'modelo__num_leaves': 15, 'modelo__n_estimators': 200, 'modelo__min_child_samples': 100, 'modelo__learning_rate': 0.05} AUC 0.6656


,params,mean_test_score,std_test_score
0,"{'modelo__reg_lambda': 0.0, 'modelo__num_leave...",0.665604,0.012613
1,"{'modelo__reg_lambda': 0.0, 'modelo__num_leave...",0.665432,0.012804
2,"{'modelo__reg_lambda': 1.0, 'modelo__num_leave...",0.665227,0.012651
3,"{'modelo__reg_lambda': 0.0, 'modelo__num_leave...",0.663891,0.012065
4,"{'modelo__reg_lambda': 0.0, 'modelo__num_leave...",0.663805,0.012035


## Decisao do modelo

A busca de hiperparametros levou a Logistica a 0,6638 e o LightGBM a 0,6656 de ROC-AUC na CV por municipio: diferenca de 0,18 ponto, abaixo do criterio de 2 pontos e do desvio entre folds (0,013). **Modelo final: Regressao Logistica (C=0,01)**, mais simples, rapida e explicavel para gestores. O LightGBM escolhendo arvores pequenas (15 folhas) indica pouca nao linearidade a explorar.

In [4]:
import joblib
from pathlib import Path
from src.evaluation.metricas import escolher_corte, avaliar, auc_por_uf

# Treino final: Logistica com o melhor C, no treino COMPLETO (1,48 milhao de alunos)
modelo_final = criar_modelos()["logistica"].set_params(modelo__C=0.01)
modelo_final.fit(Xtr, ytr, modelo__sample_weight=wtr.to_numpy())

# Corte escolhido no TREINO; o teste so e usado para medir
corte = escolher_corte(ytr, modelo_final.predict_proba(Xtr)[:, 1])
proba_te = modelo_final.predict_proba(Xte)[:, 1]

metricas_teste = pd.Series(avaliar(yte, proba_te, corte)).round(3)
print(metricas_teste)

auc_uf = auc_por_uf(yte, proba_te, Xte["sigla_uf"]).round(3)
print("\nAUC por UF no teste:")
print(auc_uf.to_string())

metricas_teste.to_csv("../reports/metricas_teste.csv", header=["valor"])
auc_uf.to_csv("../reports/auc_por_uf_teste.csv")
Path("../data/processed").mkdir(exist_ok=True)
joblib.dump(modelo_final, "../data/processed/modelo_final.joblib")

roc_auc               0.643
corte                 0.596
recall_nao_alfab      0.697
precisao_nao_alfab    0.481
recall_alfab          0.494
dtype: float64

AUC por UF no teste:
uf
AC    0.521
PB    0.522
RO    0.537
RN    0.539
MA    0.539
SE    0.539
MT    0.545
PA    0.550
MG    0.554
TO    0.558
BA    0.563
RS    0.567
AL    0.570
PR    0.574
ES    0.578
SC    0.580
SP    0.583
RJ    0.592
PE    0.609
PI    0.611
GO    0.632
AP    0.637
MS    0.639
AM    0.640
CE    0.743


['../data/processed/modelo_final.joblib']